In [ ]:
# change to the root directory of the project
import os
if os.getcwd().split("/")[-1] == "examples":
    os.chdir('..')
print(os.getcwd())

import os
import h5py
import pandas as pd


### Read data from hdf file

In [ ]:
def load_evolution_from_hdf5(hdf_path, param_names=None, max_generations=100):
    """
    Load evolution data from neurolib HDF5 file into a pandas DataFrame.
    
    Parameters
    ----------
    hdf_path : str
        Path to the HDF5 file
    param_names : list of str, optional
        Names of the parameters. If None, will use generic names.
    max_generations : int, optional
        Maximum number of generations to try loading (default: 100)
    
    Returns
    -------
    df_evol : pandas.DataFrame
        DataFrame with all individuals from all generations
    """
    
    with h5py.File(hdf_path, 'r') as f:
        # Get the trajectory name (should be the first key)
        traj_name = list(f.keys())[0]
        
        all_data = []
        
        for gen_num in range(max_generations):
            gen_key = f"gen_{gen_num:06d}"
            
            gen_path = f"{traj_name}/results/evolution/{gen_key}"
            if gen_key not in f[f"{traj_name}/results/evolution"]:
                break
                
            # Read population (parameters)
            pop = f[f"{gen_path}/population/population"][:]
            
            # Read scores
            scores = f[f"{gen_path}/scores/scores"][:]
            
            # Read fitness (multi-objective values)
            fitness = f[f"{gen_path}/fitness/fitness"][:]
            
            # Determine how many parameters to extract
            if param_names is not None:
                num_params = len(param_names)
            else:
                num_params = pop.shape[1]
            

            for i in range(len(pop)):
                row = list(pop[i][:num_params]) + [gen_num, scores[i]] + list(fitness[i])
                all_data.append(row)
        
        if param_names is None:
            param_names = [f"param_{i}" for i in range(num_params)]
        
        num_fitness = fitness.shape[1]
        fitness_cols = [f"fitness_{i}" for i in range(num_fitness)]
        columns = param_names + ["generation", "score"] + fitness_cols
        
        df_evol = pd.DataFrame(all_data, columns=columns)
        
    return df_evol


In [ ]:
hdf_path = "/mnt/raid/data/anina/ScanDy/data/hdf/new_fit/evol_bdir_obj_ll_cb_s13__std1_seed10_young.hdf"
param_names = ["ddm_thres", "ddm_sig", "att_dva", "ior_decay", "ior_inobj"]

df_evol = load_evolution_from_hdf5(hdf_path, param_names=param_names)
df_evol.sort_values("score", ascending=False)